In [ ]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client

In [ ]:
client = Client()
client = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/StoichModelGP/ModelGP_M15.json")
client.get_next_trials(max_trials=1)

In [ ]:
def PredictorsToCaStoichs(s1,b1):
    return (0.5+(2.0-0.5)*s1)*b1

def PredictorsToIaStoichs(s2,b1):
    return (1.0+(2.0-1.0)*s2)*(1-b1)

In [ ]:
trials = 300

X_lis = []
for i in range(trials):
    sampler = Sampler_class()
    Parameters_lis = [
        {"name":"x1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x3", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    for i in range(100):
        X = sampler.three.PseudorandomSampler3D_func(27,Parameters_lis).T
        if np.shape(X) != (0,3):
            break
    X_lis.append(X)

y_max_lis = []
for X in X_lis:
    s1_arr = X.T[0]
    s2_arr = X.T[1]
    b1_arr = X.T[2]
    n_ci_lis = []
    for s1,b1 in zip(s1_arr,b1_arr):
        n_ci_lis.append(PredictorsToCaStoichs(s1,b1))
    n_it_lis = []
    for s2,b1 in zip(s2_arr,b1_arr):
        n_it_lis.append(PredictorsToIaStoichs(s2,b1))
    n_ci_arr = np.array(n_ci_lis)
    n_it_arr = np.array(n_it_lis)
    y_pred_lis = []
    for n_ci,n_it in zip(n_ci_arr,n_it_arr):
        y_pred_lis.append(client.predict([{"n_ci":n_ci,"n_it":n_it}])[0]["t1"][0])
    y_max_lis.append(np.max(np.array(y_pred_lis)))

y_max_arr = np.array(y_max_lis)
print(y_max_arr.tolist())
print(np.average(y_max_arr))

In [ ]:
np.average(y_max_arr)

In [ ]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/TestPredModelGP_M15/DataGenerated/pseudorandom_27.pkl"
latestdf = pd.DataFrame(y_max_arr)
pd.to_pickle(obj=latestdf,filepath_or_buffer=filepath)